# LAYA on TriviaQA with the same state and questions as Jev

This is a new, separate LAYA run over the 9,960 saved Qwen answers and BLEURT-20 scores in notebook output `nvthaai/laya-triviaqa-gpu-multijudge`. The previous LAYA predictions are discarded. LAYA receives the exact Jev-run `assistant_policy` and formatted `user_message` state, plus the same `hallucination`, `hallucination_type`, and `severity` questions. Ground truth is joined only after inference. Enable **Internet** and **GPU T4 x2**. No API key is needed.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!pip install -q "git+https://github.com/NandhaKishorM/laya.git"

In [ ]:
from pathlib import Path
import subprocess, sys

source_csvs = [
    path for path in Path("/kaggle/input").rglob("laya_triviaqa_gpu_multijudge/results.csv")
    if "laya-triviaqa-gpu-multijudge" in path.parts
]
if len(source_csvs) != 1:
    raise RuntimeError(f"Expected one completed source result CSV, found: {source_csvs}")
SOURCE_CSV = source_csvs[0]
OUTPUT_DIR = Path("/kaggle/working/laya_triviaqa_jev_structure")
MAX_SAMPLES = 0  # 0 = all saved TriviaQA rows
command = [
    sys.executable, "/kaggle/working/SALA/laya_triviaqa.py",
    "--source-csv", str(SOURCE_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--batch-size", "8",
    "--threshold", "0.5",
    "--max-samples", str(MAX_SAMPLES),
]
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / "results.csv")
display(results.head())
print(json.dumps(json.loads((OUTPUT_DIR / "summary.json").read_text()), indent=2))
print(f"Saved {len(results):,} rows to {OUTPUT_DIR}")